# 03. Training, Fine-Tuning & Inference Pipeline
### Final Project: Multilingual Health Question Answering in Low-Resource African Languages

This notebook handles loading sequence-to-sequence models (like `mT5`), wrapping them
with PEFT LoRA adapters (EXP-07), training them, evaluating, and generating submissions.

---


In [ ]:
# Google Colab Setup (Runs automatically if in Google Colab environment)
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Detected Google Colab environment. Setting up workspace...")
    if not os.path.exists("multilingual_health_qa"):
        os.system("git clone https://github.com/belladev250/multilingual_health_qa.git")
    os.chdir("multilingual_health_qa")
    os.system("pip install -r requirements.txt")
    print("Colab workspace configured successfully!")

import pandas as pd
import numpy as np
import torch

# Align paths to make imports seamless from both notebooks/ and repository root
repo_root = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(repo_root, "src")):
    repo_root = os.path.abspath(os.path.join(repo_root, ".."))
sys.path.append(repo_root)

from src.model import load_model_and_tokenizer
from src.training import train_model
from src.inference import generate_answer, generate_submission


## 1. Setting up Model & PEFT LoRA Configuration (EXP-07)
We load the model and optionally apply parameter-efficient adapters.
This trains ~1-2% of total parameters, optimizing GPU usage and reducing overfitting.


In [ ]:
# By default, we use a tiny random model for quick verification/smoke tests.
# For full training on the real dataset, change this to "google/mt5-small" or "google/mt5-base".
MODEL_NAME = "hf-internal-testing/tiny-random-t5" 

model, tokenizer = load_model_and_tokenizer(


    model_name=MODEL_NAME,
    use_peft=True,  # Enable PEFT LoRA!
    lora_r=8,
    lora_alpha=32,
    lora_dropout=0.1
)


## 2. Preparing Datasets for Training
We load the processed datasets. If not present, we generate mock data.


In [ ]:
DATA_DIR = os.path.join(repo_root, "data/raw")

# Prioritize real uppercase files on Linux/Colab
real_train_path = os.path.join(DATA_DIR, "Train.csv")
real_test_path = os.path.join(DATA_DIR, "Test.csv")
mock_train_path = os.path.join(DATA_DIR, "train.csv")
mock_test_path = os.path.join(DATA_DIR, "test.csv")

# Clean up old mock files if real ones are present to prevent Linux case mismatch
if os.path.exists(real_train_path) and os.path.exists(mock_train_path):
    os.remove(mock_train_path)
if os.path.exists(real_test_path) and os.path.exists(mock_test_path):
    os.remove(mock_test_path)

train_path = real_train_path if os.path.exists(real_train_path) else mock_train_path

if not os.path.exists(train_path):
    from src.data_preprocessing import generate_mock_datasets
    generate_mock_datasets(DATA_DIR)
    train_path = mock_train_path

df_all = pd.read_csv(train_path)

# Train-Val split
np_random = pd.Series(range(len(df_all))).sample(frac=1.0, random_state=42)
train_indices = np_random.head(int(len(df_all) * 0.8)).index
val_indices = np_random.tail(len(df_all) - len(train_indices)).index

train_df = df_all.loc[train_indices].copy()
val_df = df_all.loc[val_indices].copy()

print(f"Data split successfully. Train size: {len(train_df)} | Val size: {len(val_df)}")


## 3. Fine-Tuning execution (EXP-01 - EXP-08)
We execute standard seq2seq training using the `train_model` engine.


In [ ]:
OUTPUT_DIR = os.path.join(repo_root, "experiments/outputs_checkpoint")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# For speed during presentation/demo, we use small limits (smoke-test slice)
# For a full real training run, increase the head slice (e.g., train_df, val_df) and epochs
train_result, eval_metrics = train_model(
    model=model,
    tokenizer=tokenizer,
    train_df=train_df.head(10), # Smoke-test slice
    val_df=val_df.head(3),
    output_dir=OUTPUT_DIR,
    epochs=1,
    lr=2e-4,
    batch_size=2,
    scheduler="cosine",
    use_prefix=True
)


## 4. Custom Generation and Decoding Strategies (EXP-10)
We generate a healthcare answer for a given question using custom generation decoders
such as **Contrastive Search**, **Nucleus (Top-p) Sampling**, and **Beam Search**.


In [ ]:
test_q = "Je, ni dalili gani za hatari wakati wa ujauzito?"
test_lang = "Kiswahili"

for dec_strat in ["greedy", "beam_search", "nucleus_sampling", "contrastive_search"]:
    ans = generate_answer(
        model=model,
        tokenizer=tokenizer,
        question=test_q,
        language=test_lang,
        use_prefix=True,
        decoding_strategy=dec_strat,
        max_length=32
    )
    print(f"\nDecoding Strategy: {dec_strat.upper()}")
    print(f"Generated Answer: '{ans}'")


## 5. Submitting Predictions to Zindi
Generating predictions for the test set and formatting the output CSV file
according to the competition instructions.


In [ ]:
real_test_path = os.path.join(DATA_DIR, "Test.csv")
mock_test_path = os.path.join(DATA_DIR, "test.csv")

test_csv_path = real_test_path if os.path.exists(real_test_path) else mock_test_path
test_df = pd.read_csv(test_csv_path)
submission_path = os.path.join(repo_root, "data/processed/zindi_submission.csv")
os.makedirs(os.path.dirname(submission_path), exist_ok=True)

sub_df = generate_submission(
    model=model,
    tokenizer=tokenizer,
    test_df=test_df,
    output_csv_path=submission_path,
    use_prefix=True,
    decoding_strategy="contrastive_search",
    max_length=32
)

print("\nSample Predictions from Submission DataFrame:")
sub_df.head()
